In [ ]:
# Load the libraries
import numpy as np
import pandas as pd
import re
import os
import matplotlib as plt

# display all row and column data
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.options.mode.copy_on_write = True

In [ ]:
from master_data_paths import *
import master_func as mfunc
import load_price_data_db as lpdb

In [ ]:
state_sector_df = mfunc.get_market_master()[['state_id', 'sector_id', 'lgd_state_name', 'nss_sector_name', 'nss_state_code', 'nss_sector_code']].drop_duplicates()
# rename lgd_state_name to state_name
state_sector_df.rename(columns={'lgd_state_name': 'state_name', 'nss_state_code': 'state_code', 'nss_sector_code': 'sector_code'}, inplace=True)

coicop_names = mfunc.get_coicop_names_codes()[['pitem_id', 'witem_id', 'subclass_id', 'class_id', 'group_id', 'division_id', 
                                               'pitem_name', 'witem_name', 'subclass_name', 'class_name', 'group_name', 
                                               'division_name', 'pitem_code', 'witem_code', 'subclass_code', 'class_code',
                                               'group_code', 'division_code']].drop_duplicates()

state_sector_df.shape, coicop_names.shape

In [ ]:
from operator import le

from numpy import int32

def get_all_level_exp(witem_exp_share, level):
    witem_exp_share = witem_exp_share.copy()
    witem_exp_share.rename(columns={'exp_share':'witem_exp'}, inplace=True)
    coicop_heirarchy_df = mfunc.get_coicop_heirarchy()
    coicop_heirarchy_df.drop(columns=['pitem_id'], inplace=True)
    coicop_heirarchy_df.drop_duplicates(inplace=True)
    
    all_levels_exp_df = witem_exp_share.merge(coicop_heirarchy_df,
                                            on='witem_id',
                                            how='left')
    total_exp = 6545371770836.336
    if level == 'witem':
        all_levels_exp_df = all_levels_exp_df[['state_id', 'sector_id', 'witem_id', 'witem_exp']]
        wts_df = all_levels_exp_df.groupby(['state_id', 'sector_id', 'witem_id'])['witem_exp'].sum().reset_index()
        wts_df.rename(columns={'witem_exp': 'witem_exp'}, inplace=True)        
    elif level == 'subclass':
         all_levels_exp_df = all_levels_exp_df[['state_id', 'sector_id', 'subclass_id', 'witem_exp']]
         wts_df = all_levels_exp_df.groupby(['state_id', 'sector_id', 'subclass_id'])['witem_exp'].sum().reset_index()
         wts_df.rename(columns={'witem_exp': 'subclass_exp'}, inplace=True)
    elif level == 'class':
        all_levels_exp_df = all_levels_exp_df[['state_id', 'sector_id', 'class_id', 'witem_exp']]
        wts_df = all_levels_exp_df.groupby(['state_id', 'sector_id', 'class_id'])['witem_exp'].sum().reset_index()
        wts_df.rename(columns={'witem_exp': 'class_exp'}, inplace=True)
    elif level == 'group':
        all_levels_exp_df = all_levels_exp_df[['state_id', 'sector_id', 'group_id', 'witem_exp']]
        wts_df = all_levels_exp_df.groupby(['state_id', 'sector_id', 'group_id'])['witem_exp'].sum().reset_index()
        wts_df.rename(columns={'witem_exp': 'group_exp'}, inplace=True)
    elif level == 'division':
        all_levels_exp_df = all_levels_exp_df[['state_id', 'sector_id', 'division_id', 'witem_exp']]
        wts_df = all_levels_exp_df.groupby(['state_id', 'sector_id', 'division_id'])['witem_exp'].sum().reset_index()
        wts_df.rename(columns={'witem_exp': 'division_exp'}, inplace=True)
    elif level == 'all':
        all_levels_exp_df = all_levels_exp_df[['state_id', 'sector_id', 'witem_exp']]
        wts_df = all_levels_exp_df.groupby(['state_id', 'sector_id'])['witem_exp'].sum().reset_index()
        wts_df.rename(columns={'witem_exp': 'all_exp'}, inplace=True)
    
    wts_df['weight'] = wts_df[f'{level}_exp']/total_exp
    # remove the column with expenditure values
    wts_df.drop(columns=[f'{level}_exp'], inplace=True)

    if level != 'all':
        combined_wts = wts_df.groupby(['state_id', f'{level}_id'])['weight'].sum().reset_index()
        combined_wts['sector_id'] = int32(3)
        wts_df = pd.concat([wts_df, combined_wts], ignore_index=True)
        all_india_wts = wts_df.groupby(['sector_id', f'{level}_id'])['weight'].sum().reset_index()
    else:
        combined_wts = wts_df.groupby(['state_id'])['weight'].sum().reset_index()
        combined_wts['sector_id'] = int32(3)
        wts_df = pd.concat([wts_df, combined_wts], ignore_index=True)
        all_india_wts = wts_df.groupby(['sector_id'])['weight'].sum().reset_index()

    all_india_wts['state_id'] = int32(0)

    # append all india weights to the combined weights dataframe
    wts_df = pd.concat([wts_df, all_india_wts], ignore_index=True)

    return wts_df

In [ ]:
# Last 13 months from price_data_month_year
if price_data_status == 'P':
    idx_export_month_count = 14
else:
    idx_export_month_count = 13
idx_month_count = 23

months_to_consider = [(price_data_month_year - pd.DateOffset(months=i)).replace(day=1) for i in range(1, idx_month_count)]
months_to_consider = [month for month in months_to_consider if month >= pd.to_datetime('2025-01-01')]
months_to_export = [(price_data_month_year - pd.DateOffset(months=i)).replace(day=1) for i in range(0, idx_export_month_count)]

In [ ]:
all_idx_df = pd.read_parquet(all_idx_db_final)

# filter the records of containing prev_year_month or price_data_month_year
# mask1 is filter for price_month_year and price_data_status from master_data_paths.py file 
mask1 = ((all_idx_df['price_month_year'] == price_data_month_year))
mask2 = ((all_idx_df['price_month_year'].isin(months_to_consider)) & (all_idx_df['price_data_status'] == "F"))
all_idx_df = all_idx_df[mask1 | mask2]

all_idx_df.head(1)

In [ ]:
all_pidx_df = pd.read_parquet(pidx_db_final)
mask1 = ((all_pidx_df['price_month_year'] == price_data_month_year))
mask2 = ((all_pidx_df['price_month_year'].isin(months_to_consider)) & (all_pidx_df['price_data_status'] == "F"))

all_pidx_df = all_pidx_df[mask1 | mask2]

all_pidx_df = all_pidx_df[['state_id', 'sector_id', 'imputed', 'pitem_id', 'price_index', 'price_month_year', 'price_data_status', 'iteration_id']]
all_pidx_df.insert(3, 'level', 'pitem')
all_pidx_df.rename(columns={'pitem_id': 'level_id', 'price_index': 'index'}, inplace=True)

all_idx_df = pd.concat([all_idx_df, all_pidx_df], ignore_index=True)
all_idx_df['level_id'] = all_idx_df['level_id'].astype('int32')
all_idx_df.head(2)

In [ ]:
from pyparsing import col


cols_req = ['state_id', 'sector_id', 'house_category', 'imputed', 'cat_index', 'price_month_year', 'price_data_status']
hr_cat_idx = pd.read_parquet(hr_cat_wise_idx_db_final)
mask1 = ((hr_cat_idx['price_month_year'] == price_data_month_year))
mask2 = ((hr_cat_idx['price_month_year'].isin(months_to_consider)) & (hr_cat_idx['price_data_status'] == "F"))
hr_cat_idx = hr_cat_idx[mask1 | mask2]
# replace the nss_sector_code with sector_id as 01 with 1 and 02 with 2
hr_cat_idx['sector_id'] = hr_cat_idx['nss_sector_code'].astype(int)
hr_cat_idx = hr_cat_idx[cols_req]
hr_cat_idx = hr_cat_idx.merge(state_sector_df, on=['state_id', 'sector_id'], how ="left")
hr_cat_idx.rename(columns={'nss_sector_name': 'sector_name'}, inplace=True)
# drop state_id and sector_id columns
#hr_cat_idx.drop(columns=['state_id', 'sector_id'], inplace=True)

cols_req = ['state_id', 'sector_id', 'total_units_price', 'nss_discom_code', 'dslab_price_index', 'price_month_year', 'price_data_status']
elect_slab_idx = pd.read_parquet(elect_dslab_pidx_db_final)
# same filter for price_month_year and price_data_status from master_data_paths.py file
mask1 = ((elect_slab_idx['price_month_year'] == price_data_month_year))
mask2 = ((elect_slab_idx['price_month_year'].isin(months_to_consider)) & (elect_slab_idx['price_data_status'] == "F"))
elect_slab_idx = elect_slab_idx[mask1 | mask2][cols_req]

elect_slab_idx = elect_slab_idx.merge(state_sector_df, how="left", on = ['state_id', 'sector_id'])
# rename nss_sector_name to sector_name
elect_slab_idx.rename(columns={'nss_sector_name': 'sector_name'}, inplace=True)

cols_req = ['state_id', 'sector_id', 'pitem_id', 'imputed', 'price_index', 'price_month_year', 'price_data_status']
pds_idx = pd.read_parquet(pds_pidx_db_final)
# same filter for price_month_year and price_data_status from master_data_paths.py file
mask1 = ((pds_idx['price_month_year'] == price_data_month_year))
mask2 = ((pds_idx['price_month_year'].isin(months_to_consider)) & (pds_idx['price_data_status'] == "F"))
pds_idx = pds_idx[mask1 | mask2][cols_req]
pds_idx = pds_idx.merge(state_sector_df, how="left", on = ['state_id', 'sector_id'])
pds_idx = pds_idx.merge(coicop_names[['pitem_id', 'pitem_name']], how="left", on = ['pitem_id'])
# rename nss_sector_name to sector_name
pds_idx.rename(columns={'nss_sector_name': 'sector_name'}, inplace=True)

cols_req = ['state_id', 'pitem_id', 'operator_name', 'imputed', 'operator_index', 'price_month_year', 'price_data_status']
telecom_idx = pd.read_parquet(telecom_operator_pidx_db_final)
# same filter for price_month_year and price_data_status from master_data_paths.py file
mask1 = ((telecom_idx['price_month_year'] == price_data_month_year))
mask2 = ((telecom_idx['price_month_year'].isin(months_to_consider)) & (telecom_idx['price_data_status'] == "F"))
telecom_idx = telecom_idx[mask1 | mask2][cols_req]
telecom_idx = telecom_idx.merge(state_sector_df[['state_id', 'state_name']].drop_duplicates(), how="left", on = ['state_id'])
telecom_idx = telecom_idx.merge(coicop_names[['pitem_id', 'pitem_name']], how="left", on = ['pitem_id'])
# rename nss_sector_name to sector_name
telecom_idx.rename(columns={'nss_sector_name': 'sector_name'}, inplace=True)
telecom_idx.head(1)


cols_req = ['slab_fare_id', 'train_type', 'class_coach', 'dist_slab', 'rep_dist_kms', 'rfare_index', 'imputed', 'price_month_year', 'price_data_status']
railfare_idx = pd.read_parquet(railfare_pidx_db_final)
# same filter for price_month_year and price_data_status from master_data_paths.py file
mask1 = ((railfare_idx['price_month_year'] == price_data_month_year))
mask2 = ((railfare_idx['price_month_year'].isin(months_to_consider)) & (railfare_idx['price_data_status'] == "F"))
railfare_idx = railfare_idx[mask1 | mask2][cols_req]
pds_idx.head(1)

In [ ]:
# check the count of records in each of the dataframes for each price_month_year and price_data_status
print("All Index Dataframe:")
print(all_idx_df.groupby(['price_month_year', 'price_data_status']).size().reset_index(name='count'))
print("\nPitem Index Dataframe:")
print(all_pidx_df.groupby(['price_month_year', 'price_data_status']).size().reset_index(name='count'))
print("\nHR Category-wise Index Dataframe:")
print(hr_cat_idx.groupby(['price_month_year', 'price_data_status']).size().reset_index(name='count'))
print("\nElectricity Slab Index Dataframe:")
print(elect_slab_idx.groupby(['price_month_year', 'price_data_status']).size().reset_index(name='count'))
print("\nPDS Index Dataframe:")
print(pds_idx.groupby(['price_month_year', 'price_data_status']).size().reset_index(name='count'))
print("\nTelecom Operator Index Dataframe:")
print(telecom_idx.groupby(['price_month_year', 'price_data_status']).size().reset_index(name='count'))
print("\nRailfare Index Dataframe:")
print(railfare_idx.groupby(['price_month_year', 'price_data_status']).size().reset_index(name='count'))

In [ ]:
# get all levels from jan_2025_idx_df and remove the price_data_status, date_updation_uid, iteration_id, price_month_year columns
# and save each level in the sheet of the excel file. Add the state, sector and coicop names to the respective dataframes

level_dfs = {}
for month, status in all_idx_df[['price_month_year', 'price_data_status']].drop_duplicates().values:
    levels = all_idx_df['level'].unique()
    for level in levels:
        print(f'Processing level: {level} for month: {month} and status: {status}')
        level_df = all_idx_df[(all_idx_df['level'] == level) & (all_idx_df['price_month_year'] == month) & (all_idx_df['price_data_status'] == status)].copy()
        #print(level_df.shape)
        level_df.reset_index(drop=True, inplace=True)
        level_df.rename(columns={'level_id': f'{level}_id', 'index': f'{level}_index'}, inplace=True)
        #print( level_df.columns)
        level_df.drop(columns=['level','price_month_year'], inplace=True)
        # merge for weights
        if level != "pitem":
            if level == "all":
                 level_df = level_df.merge(get_all_level_exp(mfunc.get_weighted_expenditure(), level=level), left_on=['state_id', 'sector_id'],
                                            right_on=['state_id', 'sector_id'], how='left')
            else:
                level_df = level_df.merge(get_all_level_exp(mfunc.get_weighted_expenditure(), level=level), left_on=['state_id', 'sector_id', f'{level}_id'],
                right_on=['state_id', 'sector_id', f'{level}_id'], how='left')

        level_df = level_df.merge(state_sector_df[['state_id', 'state_name', 'state_code']].drop_duplicates(), left_on=['state_id'], 
                                right_on=['state_id'], how='left')
        level_df = level_df.merge(state_sector_df[['sector_id', 'nss_sector_name']].drop_duplicates(), left_on=['sector_id'], 
                                right_on=['sector_id'], how='left')
        # rename nss_sector_name to sector_name
        level_df.rename(columns={'nss_sector_name': 'sector_name'}, inplace=True)
        # rename nss_state_code to state_code and nss_sector_code to sector_code
        # remove the state_id and sector_id columns
        #level_df.drop(columns=['state_id', 'sector_id'], inplace=True)
        
        # Merge with coicop_names if level is coicop related
        if level in ['pitem', 'witem', 'subclass', 'class', 'group', 'division']:
            cnames_df = coicop_names[[f'{level}_id', f'{level}_name', f'{level}_code']].drop_duplicates()
            level_df = level_df.merge(coicop_names, left_on=f'{level}_id', right_on=f'{level}_id', how='left')

            # remove the level_id columns
            level_df.drop(columns=[f'{lvl}_id' for lvl in ['pitem', 'witem', 'subclass', 'class', 'group', 'division']], inplace=True)
            level_df.drop(columns=[f'{lvl}_code' for lvl in ['pitem', 'witem', 'subclass', 'class', 'group', 'division'] if lvl != level], inplace=True)
            level_df.drop(columns=[f'{lvl}_name' for lvl in ['pitem', 'witem', 'subclass', 'class', 'group', 'division'] if lvl != level], inplace=True)

            # put the imputed columns at the end
            imputed_cols = [col for col in level_df.columns if 'imputed' in col or f'{level}_index' in col]
            non_imputed_cols = [col for col in level_df.columns if col not in imputed_cols]
            level_df = level_df[non_imputed_cols + imputed_cols]
            # rename lgd_state_name to state_name
            #level_df.rename(columns={'lgd_state_name': 'state_name'}, inplace=True)
            # replace NCT of Delhi with Delhi in state_name
            #print(level_df.columns)

        #level_df['state_name'] = level_df['state_name'].replace('NCT of Delhi', 'Delhi')
        # if state_name is null then replace with All India
        if level_df['state_name'].isnull().sum() > 0:
            level_df['state_name'] = level_df['state_name'].fillna('All India')
            level_df['state_code'] = level_df['state_code'].fillna('00')
        # if sector_name is null then replace with combined
        if level_df['sector_name'].isnull().sum() > 0:
            level_df['sector_name'] = level_df['sector_name'].fillna('Combined')
        
        id_cols = [col for col in level_df.columns if col.endswith('_id')]
        level_df.drop(columns=id_cols, inplace=True)
        level_df.drop_duplicates(inplace=True)

        # arrange columns to have state_name, sector_name, level_id, level_index at the front
        front_cols = ['state_name', 'sector_name', f'{level}_index']
        other_cols = [col for col in level_df.columns if col not in front_cols]
        level_df = level_df[front_cols + other_cols]
        
        level = level + "_" + month.strftime('%Y_%m') + "_" + status
        level_dfs[level] = level_df

In [ ]:
# append all witem , subclass, class, group, division dataframes for all months their respective dataframes
# and add a month column to each dataframe before appending from the level_df dictionary having keys as level_yyyy_mm
import stat
appended_level_dfs = {}
for level_base in ['pitem', 'witem', 'subclass', 'class', 'group', 'division', 'all']:
    temp_dfs = []
    for key in level_dfs.keys():
        if key.startswith(level_base + "_"):
            print(f'Appending data for level: {level_base} from key: {key}')
            month_str = key.split("_")[-3] + "_" + key.split("_")[-2]
            month_dt = pd.to_datetime(month_str, format='%Y_%m')
            status = key.split("_")[-1]
            if month_str == '2024_12':
                continue
            #print(f'Extracted month: {month_dt} from key: {key}')
            df_copy = level_dfs[key].copy()
            df_copy['month'] = month_dt
            df_copy['price_data_status'] = status

            temp_dfs.append(df_copy)
    appended_level_dfs[level_base] = pd.concat(temp_dfs, ignore_index=True)

In [ ]:
if 0:
    # Check for count of pitem with imputed as 1 in the appended_level_dfs['pitem'] dataframe
    imputed_counts = appended_level_dfs['pitem']
    pitem_witem_type = mfunc.get_coicop_names_codes()[['pitem_code', 'item_type_name']].drop_duplicates()
    # merge to get item_type_name
    imputed_counts = imputed_counts.merge(pitem_witem_type, how='left', on=['pitem_code'])
    # Group by state_name, nss_sector_name, month , item_type_name and count total records and sum of imputed
    imputed_summary = imputed_counts.groupby(['state_name', 'nss_sector_name', 'month', 'item_type_name']).agg(
        total_records=('imputed', 'count'),
        total_imputed=('imputed', 'sum')
    ).reset_index()
    # count percentage of imputed
    imputed_summary['imputed_percentage'] = (imputed_summary['total_imputed'] / imputed_summary['total_records']) * 100
    # Percentage of total count and imputed count for each state_name, nss_sector_name, month
    state_sector_month_totals = imputed_summary.groupby(['state_name', 'nss_sector_name', 'month']).agg(
        state_sector_total_records=('total_records', 'sum'),
        state_sector_total_imputed=('total_imputed', 'sum')
    ).reset_index()
    # Count the total imputed percentage for each state_name, nss_sector_name, month
    state_sector_month_totals['total_imputed_percentage'] = (state_sector_month_totals['state_sector_total_imputed'] / state_sector_month_totals['state_sector_total_records']) * 100
    # save to scrutiny folder
    # save into an excel with sheet name as pitem_imputed_summary and state_sector_month_totals
    with pd.ExcelWriter('data//scrutiny/pitem_imputed_summary.xlsx') as writer:
        state_sector_month_totals.to_excel(writer, sheet_name='elementary_idx_state', index=False)
        imputed_summary.to_excel(writer, sheet_name='elementary_idx_imputed', index=False)

In [ ]:
state_df = mfunc.get_market_master()[['nss_state_code', 'lgd_state_name', 'state_id']].drop_duplicates()
# concatenate a row {'nss_state_code': '00', 'lgd_state_name': 'All India'} to state_df
state_df = pd.concat([state_df, pd.DataFrame([{'nss_state_code': '00', 'lgd_state_name': 'All India', 'state_id': 0}])], ignore_index=True)
# rename lgd_state_name to state_name
state_df.rename(columns={'lgd_state_name': 'state_name'}, inplace=True)

wts_df = get_all_level_exp(mfunc.get_weighted_expenditure(), level='all')
# merge state_df with wts_df on state_id and keep state_name column
wts_df = wts_df.merge(state_df[['state_id', 'nss_state_code']].drop_duplicates(), how='left', on='state_id')
# rename nss_state_code to state_code
wts_df.rename(columns={'nss_state_code': 'state_code'}, inplace=True)
# give sector_id 1 as sector_code '01', 2 as '02' and 3 as '03'
wts_df['sector_name'] = wts_df['sector_id'].replace({1: 'Rural', 2: 'Urban', 3: 'Combined'})
# remove id columns
wts_df.drop(columns=['state_id', 'sector_id'], inplace=True)
wts_df.head(1)


In [ ]:
def get_yoy_inflation(df, key_cols, value_col, month_col):
    idx_df = df.copy()

    month_combinations = idx_df[key_cols].drop_duplicates()
    # 12 month ago month combinations
    month_combinations[month_col] = month_combinations[month_col] - pd.DateOffset(months=12)
    # set date to first day of the month
    month_combinations[month_col] = month_combinations[month_col].apply(lambda x: x.replace(day=1))

    idx_12months_ago = idx_df.merge(month_combinations, on=key_cols, how='inner')[key_cols + [f'{value_col}']]
    idx_12months_ago.rename(columns={f'{value_col}': f'{value_col}_12months_ago'}, inplace=True)
    # increase the month by 12 months to get the current month combinations
    idx_12months_ago[month_col] = idx_12months_ago[month_col] + pd.DateOffset(months=12)
    # set the date to first day of the month
    idx_12months_ago[month_col] = idx_12months_ago[month_col].apply(lambda x: x.replace(day=1))
    # merge with the original dataframe to get the current month index
    idx_df = idx_df.merge(idx_12months_ago, on=key_cols, how='left')
    # calculate yoy inflation
    idx_df[f'yoy_inflation'] = ((idx_df[f'{value_col}'] - idx_df[f'{value_col}_12months_ago']) / idx_df[f'{value_col}_12months_ago']) * 100
    idx_df.drop(columns=[f'{value_col}_12months_ago'], inplace=True)

    return idx_df

def get_mom_inflation(df, key_cols, value_col, month_col):
    idx_df = df.copy()

    month_combinations = idx_df[key_cols].drop_duplicates()
    # 1 month ago month combinations
    month_combinations[month_col] = month_combinations[month_col] - pd.DateOffset(months=1)
    # set date to first day of the month
    month_combinations[month_col] = month_combinations[month_col].apply(lambda x: x.replace(day=1))

    idx_1month_ago = idx_df.merge(month_combinations, on=key_cols, how='inner')[key_cols + [f'{value_col}']]
    idx_1month_ago.rename(columns={f'{value_col}': f'{value_col}_1month_ago'}, inplace=True)
    # increase the month by 1 month to get the current month combinations
    idx_1month_ago[month_col] = idx_1month_ago[month_col] + pd.DateOffset(months=1)
    # set the date to first day of the month
    idx_1month_ago[month_col] = idx_1month_ago[month_col].apply(lambda x: x.replace(day=1))
    # merge with the original dataframe to get the current month index
    idx_df = idx_df.merge(idx_1month_ago, on=key_cols, how='left')
    # calculate mom inflation
    idx_df[f'mom_inflation'] = ((idx_df[f'{value_col}'] - idx_df[f'{value_col}_1month_ago']) / idx_df[f'{value_col}_1month_ago']) * 100
    idx_df.drop(columns=[f'{value_col}_1month_ago'], inplace=True)

    return idx_df


In [ ]:
key_cols = ["state_id", "sector_id", "house_category", "price_month_year"]
hr_idx_df = get_mom_inflation(hr_cat_idx, key_cols, 'cat_index', 'price_month_year')
hr_yoy_df = get_yoy_inflation(hr_cat_idx, key_cols, 'cat_index', 'price_month_year')[key_cols + ['price_data_status', 'yoy_inflation']]
hr_idx_df = hr_idx_df.merge(hr_yoy_df, on=key_cols + ['price_data_status'], how='left')
# filter only price_month_year which are in list months_to_export
hr_idx_df = hr_idx_df[hr_idx_df['price_month_year'].isin(months_to_export)]

key_cols = ["state_id", "sector_id", "total_units_price", "nss_discom_code", "price_month_year"]
elect_idx_df = get_mom_inflation(elect_slab_idx, key_cols, 'dslab_price_index', 'price_month_year')
elect_yoy_df = get_yoy_inflation(elect_slab_idx, key_cols, 'dslab_price_index', 'price_month_year')[key_cols + ['price_data_status', 'yoy_inflation']]
elect_idx_df = elect_idx_df.merge(elect_yoy_df, on=key_cols + ['price_data_status'], how='left')
elect_idx_df = elect_idx_df[elect_idx_df['price_month_year'].isin(months_to_export)]

key_cols = ["state_id", "sector_id", "pitem_id", "price_month_year"]
pds_df = get_mom_inflation(pds_idx, key_cols, 'price_index', 'price_month_year')
pds_yoy_df = get_yoy_inflation(pds_idx, key_cols, 'price_index', 'price_month_year')[key_cols + ['price_data_status', 'yoy_inflation']]
pds_df = pds_df.merge(pds_yoy_df, on=key_cols + ['price_data_status'], how='left')
pds_df = pds_df[pds_df['price_month_year'].isin(months_to_export)]

key_cols = ["state_id", "pitem_id", "operator_name", "price_month_year"]
telecom_idx_df = get_mom_inflation(telecom_idx, key_cols, 'operator_index', 'price_month_year')
telecom_yoy_df = get_yoy_inflation(telecom_idx, key_cols, 'operator_index', 'price_month_year')[key_cols + ['price_data_status', 'yoy_inflation']]
telecom_idx_df = telecom_idx_df.merge(telecom_yoy_df, on=key_cols + ['price_data_status'], how='left')
telecom_idx_df = telecom_idx_df[telecom_idx_df['price_month_year'].isin(months_to_export)]


key_cols = ["slab_fare_id", "price_month_year"]
railfare_yoy_df = get_yoy_inflation(railfare_idx, key_cols, 'rfare_index', 'price_month_year')[key_cols + ['price_data_status', 'yoy_inflation']]
railfare_idx_df = get_mom_inflation(railfare_idx, key_cols, 'rfare_index', 'price_month_year')
railfare_idx_df = railfare_idx_df.merge(railfare_yoy_df, on=key_cols + ['price_data_status'], how='left')
railfare_idx_df = railfare_idx_df[railfare_idx_df['price_month_year'].isin(months_to_export)]

In [ ]:
idx_for_export = {}
idx_for_export_wide = {}
for level, df in appended_level_dfs.items():
    print(df.shape)
    if level == 'all':
        idx_cols = ['state_name', 'sector_name', 'month']
    else:
        idx_cols = ['state_name', 'sector_name', f'{level}_code', 'month']
    df1 = get_yoy_inflation(df, idx_cols, f'{level}_index', 'month')
    print(df1.shape)
    df2 = get_mom_inflation(df, idx_cols, f'{level}_index', 'month')
    print(df2.shape)

    if level == 'all':
        key_cols = ['state_name', 'sector_name', 'month', 'price_data_status']
    else:
        key_cols = ['state_name', 'sector_name', f'{level}_code', 'month', 'price_data_status']
    
    df = df1.merge(df2[key_cols + ['mom_inflation']], on=key_cols, how='left')
    # select only months which are in months_to_export
    df = df[df['month'].isin(months_to_export)]
    
    print(df.shape)
    
    # convert df into wide format with
    # sector_name, state_name, state_code, pitem_code, pitem_name, weight as key columns
    # the columns should contains the index, yoy_inflation and mom_inflation for each month and price_data_status combination
    # Create a combined column for month and status
    df['month_status'] = df['month'].dt.strftime('%b_%Y') + '_' + df['price_data_status']

    # sort by month in ascending order and price_data_status with P first and then F
    df.sort_values(by=['month', 'price_data_status'], ascending=[True, True], inplace=True)

    # Define id columns based on level
    if level == 'all':
        id_cols = ['sector_name', 'state_name', 'state_code', 'weight']
    elif level == 'pitem':
        id_cols = ['sector_name', 'state_name', 'state_code', f'{level}_code', f'{level}_name']
    else:
        id_cols = ['sector_name', 'state_name', 'state_code', f'{level}_code', f'{level}_name', 'weight']

    # rename mom_inflation and yoy_inflation columns to mom and yoy
    df.rename(columns={'yoy_inflation': 'yoy', 'mom_inflation': 'mom'}, inplace=True)

    # Pivot to wide format
    df_wide = df.pivot_table(
        index=id_cols,
        columns='month_status',
        values=[f'{level}_index', 'yoy', 'mom'],
        aggfunc='first'
    )

    # delet month_status from df and rename yoy and mom columns to yoy_inflation and mom_inflation
    df.drop(columns=['month_status'], inplace=True)
    df.rename(columns={'yoy': 'yoy_inflation', 'mom': 'mom_inflation'}, inplace=True)
    
    # arrange the columns in the order of month and price_data_status with P first and then F for each month
    month_status_order = []
    for month in sorted(df['month'].unique()):
        month_str = month.strftime('%b_%Y')
        for status in ['P', 'F']:
            month_status_order.append(f'{month_str}_{status}')
    # create a new column order based on the month_status_order and the level
    new_column_order = []
    for metric in [f'{level}_index', 'yoy', 'mom']:
        for month_status in month_status_order:
            if (metric, month_status) in df_wide.columns:
                new_column_order.append((metric, month_status))
    #print(f'New column order for level {level}: {new_column_order}')
    df_wide = df_wide[new_column_order]
    # Flatten multi-level column headers: e.g. "pitem_index_Jan_2025_P"
    df_wide.columns = [f'{val}_{ms}' for val, ms in df_wide.columns]
    df_wide.reset_index(inplace=True)
    idx_for_export_wide[level] = df_wide
    idx_for_export[level] = df

In [ ]:
# Get today's date in ddmmyyyy format
today_date = pd.to_datetime('today').strftime('%d%m%Y')
# check if the output file already exists, if yes then add today's date to the file name
output_file_name = os.path.join(output_data_path, f'{today_date}_id{iteration_id}.xlsx')
if os.path.exists(output_file_name):
    print(f'Output file {output_file_name} already exists. Adding date to the file name.')
    output_file_name = os.path.join(output_data_path, f'{today_date}_id{iteration_id}.xlsx')

# write to excel with each level in a different sheet
with pd.ExcelWriter(output_file_name, engine='openpyxl') as writer:
    for level, df_wide in idx_for_export_wide.items():
        df_wide.to_excel(writer, sheet_name=level, index=False)
    hr_idx_df.to_excel(writer, sheet_name='hr_cat_idx', index=False)
    elect_idx_df.to_excel(writer, sheet_name='elect_slab_idx', index=False)
    pds_df.to_excel(writer, sheet_name='pds_idx', index=False)
    telecom_idx_df.to_excel(writer, sheet_name='telecom_op_idx', index=False)
    railfare_idx_df.to_excel(writer, sheet_name='railfare_idx', index=False)

In [ ]:
if 0:
    import pandas as pd
    from master_func import *

    t0 = pd.read_parquet(pitem_exp_db_final)
    t1 = pd.read_parquet(witem_exp_db_final)
    t2 = pd.read_parquet(elect_weights_db_final)
    t3 = pd.read_parquet(hr_category_weights_db_final)
    t4 = pd.read_parquet(hr_own_rent_exp_share_db_final)
    t5 = get_coicop_names_codes()[["pitem_id", "pitem_name", "pitem_code", "witem_id", "witem_name", "witem_code"]].drop_duplicates()
    t6 = get_market_master()[["state_id", "sector_id", "lgd_state_name", "nss_sector_name"]].drop_duplicates()
    t7 = pd.read_parquet(pds_pitem_exp_share_db_final)
    t8 = pd.read_parquet(telecom_weights_db_final)

    # save all the dataframes to excel file with each dataframe in a different sheet
    with pd.ExcelWriter('data//output//wts_and_coicop_v04022026.xlsx', engine='openpyxl') as writer:
        t0.to_excel(writer, sheet_name='elementary_idx_wts', index=False)
        t1.to_excel(writer, sheet_name='witem_exp_share', index=False)
        t2.to_excel(writer, sheet_name='elect_discom_wts', index=False)
        t3.to_excel(writer, sheet_name='hr_cat_wts', index=False)
        t4.to_excel(writer, sheet_name='hr_own_rent_exp', index=False)
        t5.to_excel(writer, sheet_name='pitem_info', index=False)
        t6.to_excel(writer, sheet_name='state_info', index=False)
        t7.to_excel(writer, sheet_name='pds_pitem_exp_share', index=False)
        t8.to_excel(writer, sheet_name='telecom_operator_wts', index=False)